# **Purpose**

**Task:** context + question + correct answer → distractor (an incorrect but plausible option)

This notebook fine-tunes `facebook/bart-base` on a subset of RACE to generate a
distractor given a reading-comprehension context, its question, and the
correct answer. It's designed as a reusable template — swap `MODEL_NAME` and
rerun to benchmark other backbones (T5-base, BART-base, Flan-T5-base all use
this exact notebook, just with `MODEL_NAME`/`OUTPUT_DIR` changed).

**Runtime:** Kaggle Notebook Settings → Accelerator → GPU (T4 is enough for
`facebook/bart-base` with the settings below). Also turn **Internet** ON, since we
need it to download RACE and the base checkpoint, and to push the fine-tuned
model to the Hugging Face Hub.

## **Install dependencies**

In [1]:
!pip install -qU \
 transformers==5.16.1 \
 sentencepiece==0.2.2 \
 accelerate==1.14.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 76.6 MB/s eta 0:00:00


## **Imports**

In [2]:
import numpy as np
import torch
import transformers
import accelerate
import sentencepiece as spm

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

from kaggle_secrets import UserSecretsClient
import os

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("SentencePiece:", spm.__version__)

torch: 2.10.0+cu128
transformers: 5.16.1
accelerate: 1.14.0
SentencePiece: 0.2.2


## **Load the API Keys and Tokens**

In [3]:
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")   # must match the exact secret name you set in Kaggle Secrets

os.environ["HF_TOKEN"] = hf_token

## **Config**

Tweak these for your experiment. `TRAIN_SUBSET_SIZE` / `VAL_SUBSET_SIZE`
control how many **RACE questions** you use — each question expands into up
to 3 (context, question, correct answer) → distractor training pairs, since
RACE gives 3 human-written distractors per question. Start small to
sanity-check the pipeline before scaling up.

In [4]:
MODEL_NAME = "facebook/bart-base"             # swap this to compare T5-base / BART-base / Flan-T5-base
MAX_INPUT_LENGTH = 512                  # context+question+answer prompt length
MAX_TARGET_LENGTH = 48                  # distractors are short phrases
TRAIN_SUBSET_SIZE = 8000                # subset of RACE train questions, set to None for full data
VAL_SUBSET_SIZE = 10
TRAIN_EPOCH_SIZE = 3
OUTPUT_DIR = "/kaggle/working/bart-base-dg"
SEED = 42

# Change the Huggingface push directory below

## **Load RACE and take a subset**

Uses the Hugging Face `race` dataset, `"all"` config (High School + Middle
School questions). Swap to `"high"` or `"middle"` if you want a single
difficulty band.

In [5]:
raw = load_dataset("race", "all")

train_ds = raw["train"].shuffle(seed=SEED)
val_ds = raw["validation"].shuffle(seed=SEED)

if TRAIN_SUBSET_SIZE:
    train_ds = train_ds.select(range(TRAIN_SUBSET_SIZE))
if VAL_SUBSET_SIZE:
    val_ds = val_ds.select(range(VAL_SUBSET_SIZE))

print(train_ds)
print(val_ds)

README.md: 0.00B [00:00, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/2.08M [00:00<?, ?B/s]

all/train-00000-of-00001.parquet:   0%|          | 0.00/37.4M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/2.05M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4934 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/87866 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4887 [00:00<?, ? examples/s]

Dataset({
    features: ['example_id', 'article', 'answer', 'question', 'options'],
    num_rows: 8000
})
Dataset({
    features: ['example_id', 'article', 'answer', 'question', 'options'],
    num_rows: 10
})


In [6]:
# Peek at one example
train_ds[0]

{'example_id': 'middle6878.txt',
 'article': 'THE human face doesn\'t lie. We show sadness and happiness through our expressions. But exactly how many emotions can our face make?Scientists used to believe we had six basic facial expressions that tell others how we feel: sad, happy, surprised, fearful, angry and disgusted  .\nBut a new study shows that our faces can do more than we think. Scientists from Ohio State University found out that humans can actually make 21 different facial expressions after studying how people move their facial muscles.\nThe scientists took pictures of 230 volunteers making faces in response to different cues  .These cues included phrases like "you just got some great unexpected news", which produced a "happily surprised" reaction from volunteers. Other cues included "you smell a bad odor  ", which caused "disgusted" faces.\nIn total, around 5,000 pictures were taken of the volunteers. The scientists then studied similarities of these pictures using a comput

## **Load tokenizer and model**

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

## **Preprocessing**

Builds the prompt. Each RACE question has 3 incorrect options (distractors),
so we expand every question into up to 3 training pairs — one per gold
distractor — rather than a single deterministic target. This lets the model
learn a distribution over plausible wrong answers instead of memorizing one
fixed output per question.

**Tip:** the prompt puts the correct answer and question BEFORE the context,
so truncation (if the context is long) only ever eats into the context, never
the answer/question the model needs most.

In [8]:
LETTER_TO_IDX = {"A": 0, "B": 1, "C": 2, "D": 3}

def preprocess(examples):
    input_texts, target_texts = [], []

    for article, question, options, answer in zip(
        examples["article"], examples["question"], examples["options"], examples["answer"]
    ):
        if answer not in LETTER_TO_IDX:
            continue
        correct_idx = LETTER_TO_IDX[answer]
        if correct_idx >= len(options):
            continue
        correct_answer = options[correct_idx]

        for i, option in enumerate(options):
            if i == correct_idx or not option.strip():
                continue

            prompt = (
                f"Correct Answer: {correct_answer}\n"
                f"Question: {question}\n"
                f"Generate a plausible but incorrect answer option (a distractor) for the question "
                f"above, based on the context below. The distractor must be clearly wrong but related "
                f"to the topic, so that a student with a misconception could mistake it for the "
                f"correct answer.\n"
                f"Context: {article}"
            )
            input_texts.append(prompt)
            target_texts.append(option)

    model_inputs = tokenizer(input_texts, max_length=MAX_INPUT_LENGTH, truncation=True)
    labels = tokenizer(text_target=target_texts, max_length=MAX_TARGET_LENGTH, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True)

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

## **Training arguments**

On Kaggle's T4 GPUs, `fp16=True` (mixed precision) is set automatically when
a GPU is available. Adjust batch size / gradient accumulation if you hit
out-of-memory errors.

In [9]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    save_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    weight_decay=0.01,
    num_train_epochs=TRAIN_EPOCH_SIZE,
    fp16=torch.cuda.is_available(),   # mixed precision on GPU
    logging_steps=50,
    save_total_limit=2,
    report_to="none",   # set to "wandb"/"tensorboard" if you use experiment tracking
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    processing_class=tokenizer,
    data_collator=data_collator,
)

## **Train the model**

In [10]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
50,13.127277
100,11.663998
150,11.456233
200,11.265253
250,11.067643
300,11.176493
350,10.916740
400,10.933147
450,10.681580
500,10.819476


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2250, training_loss=7.993160298665365, metrics={'train_runtime': 2834.1061, 'train_samples_per_second': 25.405, 'train_steps_per_second': 0.794, 'total_flos': 2.1930743697408e+16, 'train_loss': 7.993160298665365, 'epoch': 3.0})

## **Save the fine-tuned model**

In [11]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved to {OUTPUT_DIR}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to /kaggle/working/bart-base-dg


## **Push the model to Huggingface Hub**

In [12]:
model.push_to_hub("gaurav-dey/bart-base-dg")
tokenizer.push_to_hub("gaurav-dey/bart-base-dg")

# model.push_to_hub("gauravdey2024/bart-base-dg")
# tokenizer.push_to_hub("gauravdey2024/bart-base-dg")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/gaurav-dey/bart-base-dg/commit/8df35c5291718e6307309cd7306d184c2c6d731f', commit_message='Upload tokenizer', commit_description='', oid='8df35c5291718e6307309cd7306d184c2c6d731f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/gaurav-dey/bart-base-dg', endpoint='https://huggingface.co', repo_type='model', repo_id='gaurav-dey/bart-base-dg'), pr_revision=None, pr_num=None)

Optional: `/kaggle/working/` is automatically persisted as this notebook's
**Output** once the session ends, so the saved checkpoint above will still be
downloadable / usable as an Input dataset for the matching evaluation
notebook even after the session disconnects.

## **Sanity-check generations**

In [13]:
sample = val_ds.select(range(5))
for ex in sample:
    answer_letter = ex["answer"]
    if answer_letter not in LETTER_TO_IDX:
        continue
    correct_idx = LETTER_TO_IDX[answer_letter]
    options = ex["options"]
    correct_answer = options[correct_idx]

    prompt = (
        f"Correct Answer: {correct_answer}\n"
        f"Question: {ex['question']}\n"
        f"Generate a plausible but incorrect answer option (a distractor) for the question "
        f"above, based on the context below. The distractor must be clearly wrong but related "
        f"to the topic, so that a student with a misconception could mistake it for the "
        f"correct answer.\n"
        f"Context: {ex['article']}"
    )

    input_ids = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LENGTH
    ).input_ids.to(model.device)

    output_ids = model.generate(input_ids, max_length=MAX_TARGET_LENGTH, num_beams=4)
    generated_distractor = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    gold_distractors = [o for i, o in enumerate(options) if i != correct_idx]

    print(f"Correct answer:       {correct_answer}")
    print(f"Gold distractors:     {gold_distractors}")
    print(f"Generated distractor: {generated_distractor}")
    print("-" * 80)

Correct answer:       his mother didn't turn up at the party as she had promised
Gold distractors:     ["he couldn't go to the party he had been looking forward to", 'his mother had refused to make chocolate chips for the party', 'the cookies his mom made was not popular at the party']
Generated distractor: his mother didn't attend the party as she had promised
--------------------------------------------------------------------------------
Correct answer:       for cleaning the bottom of shoes
Gold distractors:     ['used as a door', 'put up on a door as an ornament', 'near a door under which people put their keys']
Generated distractor: for finding the key
--------------------------------------------------------------------------------
Correct answer:       vision-phone
Gold distractors:     ['radio', 'telephone', 'television']
Generated distractor: Vision-phones
--------------------------------------------------------------------------------
Correct answer:       A charming table la

## **Next steps**

- Swap `MODEL_NAME` between `t5-base`, `facebook/bart-base`, and
  `google/flan-t5-base` (this notebook, unchanged apart from the Config cell)
  to produce all three fine-tuned checkpoints for comparison.
- Add the answer-highlighting / knowledge-graph-augmented prompt variant and
  compare against this baseline.
- The matching evaluation notebook now includes a distractor-vs-answer
  semantic similarity metric (plausibility) once this checkpoint is pushed
  to the Hub.